# Full-Graph Deterministic Evaluation V2 — Independent source F1 + temporal diagnostics

**Purpose:** Validate all exported graph nodes and links across 30 projects, audit temporal anomalies, and compute two clearly separated event reconstruction F1 scores. The primary *source-to-graph* benchmark reconstructs explicitly defined Jira changelog events from original curated tickets. The secondary *extraction-to-graph* score measures retention of already-extracted events, **not independent semantic accuracy**.

**Safeguards:** Never edit the original graph, references or labels. Project-instance eligibility is defined by the snapshot manifests, not graph predictions. Event vocabulary is fixed before scoring. One-to-one multiset matching prevents duplicate predictions earning multiple TPs. Missing inputs cause a `NOT EVALUABLE` result, not a zero score. Exact-time F1 is primary; ±1 second is a diagnostic. Full-graph structural pass rates are **not F1**.

**Caution:** The original Jira dataset is a curated sample, not necessarily every Jira ticket in existence. Thus source-to-graph precision is conditional on this curated input and the declared supported event types. does not establish generalization to all Jira events or all graph facts.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, collections, hashlib, math
import pandas as pd
import numpy as np
from IPython.display import display
ROOT=Path('/content/drive/MyDrive')
GC1=ROOT/'GC1'
JIRA=GC1/'GC1_Jira_Dataset'
SNAP=GC1/'GC1_Generated_PRD_Meetings'
RESOLVED=GC1/'entity_resolution_output'/'resolved_extraction_records.jsonl'
GRAPH=GC1/'entity_resolution_output'/'temporal_kg_v2_output'
BASE=ROOT/'01_Knowledge_Construction'/'Temporal_KG_Evaluation'
OUT=BASE/'Results'/'Full_Graph_Deterministic_V2'
OUT.mkdir(parents=True,exist_ok=True)
for label,p in [('Jira',JIRA),('Manifests',SNAP),('Resolved',RESOLVED),('Graph',GRAPH)]:
    print(label, 'FOUND' if p.exists() else 'MISSING',p)


Mounted at /content/drive
Jira FOUND /content/drive/MyDrive/GC1/GC1_Jira_Dataset
Manifests FOUND /content/drive/MyDrive/GC1/GC1_Generated_PRD_Meetings
Resolved FOUND /content/drive/MyDrive/GC1/entity_resolution_output/resolved_extraction_records.jsonl
Graph FOUND /content/drive/MyDrive/GC1/entity_resolution_output/temporal_kg_v2_output


In [2]:
def stream_jsonl(path):
    with open(path,encoding='utf-8-sig') as f:
        for i,line in enumerate(f,1):
            if line.strip():
                try: yield json.loads(line)
                except Exception as e: raise ValueError(f'{path} line {i}: {e}')
def parse_ts(value):
    if value is None or str(value).strip().lower() in {'','none','null','nan','nat','unknown'}: return pd.NaT
    return pd.to_datetime(value,utc=True,format='mixed',errors='coerce')
def stamp(value):
    t=parse_ts(value)
    return t.isoformat() if pd.notna(t) else None
def metric(tp,fp,fn):
    precision=tp/(tp+fp) if tp+fp else float('nan')
    recall=tp/(tp+fn) if tp+fn else float('nan')
    f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else float('nan')
    return dict(TP=tp,FP=fp,FN=fn,precision=precision,recall=recall,F1=f1)


## 1. Independent structural and temporal audit

Counts duplicate IDs, dangling endpoints, node/edge types, malformed dates, inverted validity intervals and availability preceding event time. Save detailed anomaly rows (capped only for display, not counting). An `observed_at` preceding `event_time` is **suspicious**, not automatically wrong: some metadata describes prior knowledge.

In [3]:
node_path=list(GRAPH.rglob('nodes.jsonl')) if GRAPH.is_dir() else []
link_path=list(GRAPH.rglob('links.jsonl')) if GRAPH.is_dir() else []
assert len(node_path)==1 and len(link_path)==1,'Expected exactly one nodes.jsonl and links.jsonl under GRAPH'
node_ids=set(); labels={}; nc=collections.Counter(); ec=collections.Counter(); anomalies=[]
seen_node_dups=0; interval_count=0; interval_reversed=0; obs_compared=0; obs_before=0
observed_by_label=collections.Counter(); missing_by_label=collections.Counter()
for x in stream_jsonl(node_path[0]):
    uid=x.get('uid'); lab=x.get('label','UNKNOWN');nc[lab]+=1
    if not uid or uid in node_ids:seen_node_dups+=1
    if uid:node_ids.add(uid);labels[uid]=lab
    if 'observed_at' in x:
        if pd.notna(parse_ts(x.get('observed_at'))):observed_by_label[lab]+=1
        else:missing_by_label[lab]+=1
    if x.get('valid_from') and x.get('valid_to'):
        a=parse_ts(x['valid_from']);z=parse_ts(x['valid_to'])
        if pd.notna(a) and pd.notna(z):
            interval_count+=1
            if a>z:
                interval_reversed+=1;anomalies.append({'kind':'REVERSED_INTERVAL','uid':uid,'label':lab,'first':str(a),'second':str(z)})
    a=parse_ts(x.get('event_time'));z=parse_ts(x.get('observed_at'))
    if pd.notna(a) and pd.notna(z):
        obs_compared+=1
        if z<a:
            obs_before+=1;anomalies.append({'kind':'OBSERVED_BEFORE_EVENT','uid':uid,'label':lab,'first':str(a),'second':str(z)})
edge_ids=set();duplicate_edges=0;dangling=0;has_event_bad=0
for x in stream_jsonl(link_path[0]):
    uid=x.get('uid');typ=x.get('type','UNKNOWN');ec[typ]+=1
    if not uid or uid in edge_ids:duplicate_edges+=1
    if uid:edge_ids.add(uid)
    a=x.get('from_uid');z=x.get('to_uid')
    if a not in node_ids or z not in node_ids:dangling+=1
    if typ=='HAS_EVENT' and labels.get(z)!='Event':has_event_bad+=1
structural={'nodes':sum(nc.values()),'edges':sum(ec.values()),'duplicate_or_missing_node_uids':seen_node_dups,'duplicate_or_missing_edge_uids':duplicate_edges,'dangling_edges':dangling,'has_event_bad_targets':has_event_bad,'valid_intervals_evaluable':interval_count,'reversed_intervals':interval_reversed,'availability_event_pairs_evaluable':obs_compared,'observed_before_event':obs_before}
print('Structural audit:');display(pd.DataFrame([structural]))
pd.DataFrame([structural]).to_csv(OUT/'structural_temporal_summary.csv',index=False)
pd.DataFrame(anomalies,columns=['kind','uid','label','first','second']).to_csv(OUT/'temporal_anomalies.csv',index=False)
coverage=pd.DataFrame([{'node_type':k,'nodes':nc[k],'observed_at_parseable':observed_by_label[k],'observed_at_missing_or_invalid_when_field_present':missing_by_label[k]} for k in sorted(nc)])
display(coverage);coverage.to_csv(OUT/'availability_by_node_type.csv',index=False)


Structural audit:


,nodes,edges,duplicate_or_missing_node_uids,duplicate_or_missing_edge_uids,dangling_edges,has_event_bad_targets,valid_intervals_evaluable,reversed_intervals,availability_event_pairs_evaluable,observed_before_event
0,206634,600623,0,0,0,0,267,35,85943,68


,node_type,nodes,observed_at_parseable,observed_at_missing_or_invalid_when_field_present
0,ActionItem,207,207,0
1,Blocker,443,167,276
2,CanonicalEntity,40325,5926,0
3,Decision,414,414,0
4,Dependency,7709,237,7472
5,Event,82577,82396,181
6,Evidence,5827,67,5760
7,Fact,63305,3809,59496
8,SourceRecord,5827,67,5760


## 2. Load instance windows from project-instance manifests

Only `PROJECT-I01`/`PROJECT-I02` manifests with `instance_start` and `instance_end` are eligible. Project-level snapshot manifests are not instance windows. A ticket is eligible if at least one recognized activity timestamp falls within its instance window. For event scoring, **the event timestamp itself must also be within that window**.

In [4]:
windows={}; manifest_errors=[]
for p in SNAP.glob('*_*/*/snapshot_manifest.json'):
    try:
        d=json.loads(p.read_text(encoding='utf-8'))
        instance=d.get('instance_id');start=parse_ts(d.get('instance_start'));end=parse_ts(d.get('instance_end'))
        if instance and pd.notna(start) and pd.notna(end) and start<=end:
            windows[instance]=(start,end)
        else:manifest_errors.append(str(p))
    except Exception as ex:manifest_errors.append(str(p)+':'+str(ex))
print('Valid instance windows:',len(windows),'manifest problems:',len(manifest_errors))
print('Examples:',list(windows.items())[:3])
assert windows,'No instance windows discovered'


Valid instance windows: 55 manifest problems: 0
Examples: [('JBEAP-I01', (Timestamp('2015-11-24 00:00:00+0000', tz='UTC'), Timestamp('2016-08-01 23:59:59.999999+0000', tz='UTC'))), ('JBEAP-I02', (Timestamp('2017-01-17 00:00:00+0000', tz='UTC'), Timestamp('2017-09-25 23:59:59.999999+0000', tz='UTC'))), ('JBIDE-I01', (Timestamp('2020-06-27 00:00:00+0000', tz='UTC'), Timestamp('2021-03-05 23:59:59.999999+0000', tz='UTC')))]


## 3. Fixed, independently specified event mapping

These are direct Jira changelog-field correspondences only. We intentionally do **not** infer `REOPENED`, `ISSUE_RESOLVED` or `ISSUE_CREATED` from other events. We treat `fixVersion` and `versions` separately. A field/type pair must agree on the specific field. `VERSION_CHANGE` alone is insufficient to imply a fix-version change.

The source benchmark is defined from `full_changelog[].items` with `full_changelog[].created`. It does not use graph predictions to choose supported categories.

In [5]:
def normalize_field(s):
    return ''.join(ch for ch in str(s or '').lower() if ch.isalnum())
FIELD_TO_TYPE={
 'status':'STATUS_CHANGED','assignee':'ASSIGNEE_CHANGED',
 'fixversion':'FIX_VERSION_CHANGED','fixversions':'FIX_VERSION_CHANGED',
 'component':'COMPONENT_CHANGED','components':'COMPONENT_CHANGED',
 'priority':'PRIORITY_CHANGED','duedate':'DUE_DATE_CHANGED',
 'duedatefield':'DUE_DATE_CHANGED'
}
# Fixed supported vocabulary, regardless of what appears in graph.
SUPPORTED=set(FIELD_TO_TYPE.values())
GRAPH_DIRECT={'STATUS_CHANGE':'STATUS_CHANGED','ASSIGNEE_CHANGE':'ASSIGNEE_CHANGED',
 'COMPONENT_CHANGE':'COMPONENT_CHANGED','PRIORITY_CHANGE':'PRIORITY_CHANGED',
 'DUE_DATE_CHANGE':'DUE_DATE_CHANGED'}
def graph_event_type(raw,field):
    raw=str(raw or '').upper().strip();f=normalize_field(field)
    if raw in GRAPH_DIRECT:return GRAPH_DIRECT[raw]
    if raw=='VERSION_CHANGE' and f in {'fixversion','fixversions'}:return 'FIX_VERSION_CHANGED'
    return None
print('Supported source types:',sorted(SUPPORTED))


Supported source types: ['ASSIGNEE_CHANGED', 'COMPONENT_CHANGED', 'DUE_DATE_CHANGED', 'FIX_VERSION_CHANGED', 'PRIORITY_CHANGED', 'STATUS_CHANGED']


## 4. Reconstruct source Jira changelog events across all projects

A record may be eligible for multiple instance windows. The unit of evaluation is `(instance_id, issue_key, event_type, UTC timestamp)`; each original changelog item counts once. Source events are limited to the declared fields and each instance window. Report any missing source files or malformed timestamp counts; do not silently convert missing sources to negatives.

In [6]:
source_counts=collections.Counter(); eligible_issue_keys=set(); input_files=0; bad_source_ts=0; raw_ticket_count=0
source_issues=collections.Counter(); source_by_project=collections.Counter(); unavailable_projects=[]
for folder in sorted(JIRA.glob('*_*')):
    if not folder.is_dir():continue
    project=folder.name.split('_',1)[-1]
    eligible_instances=[(i,w) for i,w in windows.items() if i.startswith(project+'-I')]
    if not eligible_instances:continue
    files=list(folder.glob('jira_tickets.jsonl'))
    if len(files)!=1:
        unavailable_projects.append(project);continue
    input_files+=1
    for ticket in stream_jsonl(files[0]):
        raw_ticket_count+=1
        key=str(ticket.get('issue_key') or '').upper().strip()
        if not key:continue
        activity=[]
        for fld in ['created_at','updated_at','resolved_at','due_at','blocked_at','unblocked_at']:
            t=parse_ts(ticket.get(fld))
            if pd.notna(t):activity.append(t)
        for item in ticket.get('status_history') or []:
            t=parse_ts(item.get('changed_at'))
            if pd.notna(t):activity.append(t)
        histories=ticket.get('full_changelog') or []
        for h in histories:
            t=parse_ts(h.get('created'))
            if pd.notna(t):activity.append(t)
        for inst,(start,end) in eligible_instances:
            if not any(start<=t<=end for t in activity):continue
            eligible_issue_keys.add((inst,key));source_issues[inst]+=1
            for h in histories:
                t=parse_ts(h.get('created'))
                if pd.isna(t):
                    if h.get('created'):bad_source_ts+=1
                    continue
                if not start<=t<=end:continue
                for item in h.get('items') or []:
                    typ=FIELD_TO_TYPE.get(normalize_field(item.get('field')))
                    if typ:
                        source_counts[(inst,key,typ,t.isoformat())]+=1
                        source_by_project[project]+=1
print('Curated source files:',input_files,'tickets read:',raw_ticket_count)
print('Eligible issue-instance pairs:',len(eligible_issue_keys),'source events in supported types:',sum(source_counts.values()))
print('Missing/ambiguous project inputs:',unavailable_projects,'invalid changelog timestamps:',bad_source_ts)
if unavailable_projects:print('WARNING: source-to-graph F1 will be NOT EVALUABLE until inputs are available.')


Curated source files: 30 tickets read: 9000
Eligible issue-instance pairs: 9502 source events in supported types: 32202
Missing/ambiguous project inputs: [] invalid changelog timestamps: 0


## 5. Parse graph Event nodes and align only supported fields

Only graph events whose source ticket exists in the curated eligible issue-instance set, whose event time falls inside that instance window, and whose type is in the fixed mapping enter the comparison. All excluded graph records are counted in a coverage audit, not silently discarded.

In [7]:
graph_counts=collections.Counter(); graph_raw=collections.Counter(); excluded=collections.Counter(); graph_rows=0; graph_scope=0
for x in stream_jsonl(node_path[0]):
    if x.get('label')!='Event':continue
    graph_rows+=1
    inst=str(x.get('instance_id') or '');key=str(x.get('source_id') or '').upper().strip()
    if (inst,key) not in eligible_issue_keys:
        excluded['not_in_curated_eligible_issue_set']+=1;continue
    win=windows.get(inst)
    if not win:
        excluded['unknown_instance_window']+=1;continue
    t=parse_ts(x.get('event_time'))
    if pd.isna(t):excluded['invalid_event_time']+=1;continue
    if not win[0]<=t<=win[1]:excluded['outside_instance_window']+=1;continue
    graph_scope+=1
    try:p=json.loads(x.get('payload_json') or '{}')
    except Exception:excluded['invalid_payload_json']+=1;continue
    raw=str(p.get('event_type') or '').upper();graph_raw[raw]+=1
    field=(p.get('attributes') or {}).get('jira_field')
    typ=graph_event_type(raw,field)
    if not typ:excluded['not_in_fixed_supported_vocabulary']+=1;continue
    graph_counts[(inst,key,typ,t.isoformat())]+=1
print('Graph Event nodes:',graph_rows,'eligible/in-window:',graph_scope,'comparable:',sum(graph_counts.values()))
print('Excluded breakdown:',dict(excluded))
pd.DataFrame([{'reason':k,'count':v} for k,v in excluded.items()]).to_csv(OUT/'graph_exclusion_audit.csv',index=False)


Graph Event nodes: 82577 eligible/in-window: 82053 comparable: 19657
Excluded breakdown: {'not_in_fixed_supported_vocabulary': 62396, 'not_in_curated_eligible_issue_set': 524}


## 6. Primary independent source-to-graph F1 (strict exact UTC timestamp)

Multiset TP = `sum(min(source_count, graph_count))`; FP and FN are computed from unmatched multiplicities. This is **conditional on the curated source dataset and fixed supported event types**, not open-world graph precision.

In [8]:
if unavailable_projects or bad_source_ts:
    print('NOT EVALUABLE: missing curated input files or unparseable source timestamps; inspect audit.')
    independent=None
else:
    tp=sum((source_counts & graph_counts).values())
    fp=sum((graph_counts-source_counts).values())
    fn=sum((source_counts-graph_counts).values())
    independent=metric(tp,fp,fn)
    print('Independent Jira changelog -> graph event matching:')
    display(pd.DataFrame([independent]))
    pd.DataFrame([independent]).to_csv(OUT/'source_to_graph_exact_f1.csv',index=False)
    by_type=[]
    for typ in sorted(SUPPORTED):
        s=collections.Counter({k:v for k,v in source_counts.items() if k[2]==typ})
        g=collections.Counter({k:v for k,v in graph_counts.items() if k[2]==typ})
        t=sum((s&g).values());f=sum((g-s).values());miss=sum((s-g).values())
        by_type.append({'type':typ,'source_events':sum(s.values()),'graph_events':sum(g.values()),**metric(t,f,miss)})
    display(pd.DataFrame(by_type))
    pd.DataFrame(by_type).to_csv(OUT/'source_to_graph_by_type.csv',index=False)


Independent Jira changelog -> graph event matching:


,TP,FP,FN,precision,recall,F1
0,11539,8118,20663,0.587017,0.358332,0.445014


,type,source_events,graph_events,TP,FP,FN,precision,recall,F1
0,ASSIGNEE_CHANGED,3910,2206,1398,808,2512,0.633726,0.357545,0.457162
1,COMPONENT_CHANGED,1788,1388,878,510,910,0.632565,0.491051,0.552897
2,DUE_DATE_CHANGED,6,2,0,2,6,0.000000,0.000000,0.000000
3,FIX_VERSION_CHANGED,6105,3469,2693,776,3412,0.776304,0.441114,0.562565
4,PRIORITY_CHANGED,577,364,246,118,331,0.675824,0.426343,0.522848
5,STATUS_CHANGED,19816,12228,6324,5904,13492,0.517174,0.319136,0.394707


## 7. ±1-second one-to-one sensitivity diagnostic

This is **not** the primary score. It pairs events greedily in sorted time order within each `(instance, issue, type)` group, using each event occurrence at most once. For exact optimal matching within a tolerance on one-dimensional timestamps, sorted two-pointer matching maximizes the number of pairs.

In [9]:
def tolerant_tp(a,b,seconds=1):
    aa=collections.defaultdict(list);bb=collections.defaultdict(list)
    for (inst,key,typ,ts),count in a.items():aa[(inst,key,typ)].extend([pd.Timestamp(ts)]*count)
    for (inst,key,typ,ts),count in b.items():bb[(inst,key,typ)].extend([pd.Timestamp(ts)]*count)
    tp=0
    for group in set(aa)|set(bb):
        x=sorted(aa[group]);y=sorted(bb[group]);i=j=0
        while i<len(x) and j<len(y):
            delta=(x[i]-y[j]).total_seconds()
            if abs(delta)<=seconds:tp+=1;i+=1;j+=1
            elif delta<0:i+=1
            else:j+=1
    return tp
if independent is not None:
    tolerant=tolerant_tp(source_counts,graph_counts,1)
    diagnostic=metric(tolerant,sum(graph_counts.values())-tolerant,sum(source_counts.values())-tolerant)
    print('±1-second sensitivity (not primary):');display(pd.DataFrame([diagnostic]))
    pd.DataFrame([diagnostic]).to_csv(OUT/'source_to_graph_1s_sensitivity.csv',index=False)


±1-second sensitivity (not primary):


,TP,FP,FN,precision,recall,F1
0,19656,1,12546,0.999949,0.610397,0.758055


## 8. Secondary extraction-to-graph retention F1

This score tests fidelity to the resolved extraction records, which were also inputs to the graph builder. High retention does **not** prove that the extracted events are true. All resolved Event records (Jira and non-Jira) are included by their `instance_id`, `source_id`, event type and event time. Exact schema equivalence is checked before interpreting the score.

In [10]:
resolved_events=collections.Counter(); graph_events_full=collections.Counter(); extraction_bad=0
if RESOLVED.is_file():
    for x in stream_jsonl(RESOLVED):
        src=x.get('source') or {};inst=str(x.get('instance_id') or '');sid=str(src.get('source_id') or '')
        for e in x.get('events') or []:
            typ=str(e.get('event_type') or '');ts=stamp(e.get('event_time'))
            if not typ or not ts:extraction_bad+=1;continue
            resolved_events[(inst,sid,typ,ts)]+=1
    for x in stream_jsonl(node_path[0]):
        if x.get('label')!='Event':continue
        try:p=json.loads(x.get('payload_json') or '{}')
        except Exception:p={}
        typ=str(p.get('event_type') or '');ts=stamp(x.get('event_time'))
        if typ and ts:graph_events_full[(str(x.get('instance_id') or ''),str(x.get('source_id') or ''),typ,ts)]+=1
    t=sum((resolved_events&graph_events_full).values());f=sum((graph_events_full-resolved_events).values());miss=sum((resolved_events-graph_events_full).values())
    retention={'extracted_events_scorable':sum(resolved_events.values()),'graph_events_scorable':sum(graph_events_full.values()),'extraction_missing_type_or_time':extraction_bad,**metric(t,f,miss)}
    print('Resolved extraction -> graph retention (not independent accuracy):');display(pd.DataFrame([retention]))
    pd.DataFrame([retention]).to_csv(OUT/'extraction_to_graph_retention_f1.csv',index=False)
else:
    print('NOT EVALUABLE: resolved extraction file not found')


Resolved extraction -> graph retention (not independent accuracy):


,extracted_events_scorable,graph_events_scorable,extraction_missing_type_or_time,TP,FP,FN,precision,recall,F1
0,82577,82372,0,34881,47491,47696,0.423457,0.422406,0.422931


## 9. Source discrepancies and reporting summary

Report the independent event F1, extraction-retention F1, structural pass rates, temporal anomalies and unknown availability **separately**. Do not average them into an overall graph F1. A successful integrity test does not establish semantic correctness or leakage-free risk prediction.

In [11]:
report={
 'evaluation':'Full-Graph Deterministic V2',
 'scope':'All available project instances with curated Jira inputs and manifest windows',
 'structural':structural,
 'source_to_graph_exact':independent,
 'source_to_graph_1s_diagnostic':diagnostic if independent is not None else None,
 'extraction_to_graph_retention':retention if RESOLVED.is_file() else None,
 'source_files':input_files,'source_tickets':raw_ticket_count,
 'eligible_issue_instance_pairs':len(eligible_issue_keys),
 'eligible_supported_source_events':sum(source_counts.values()),
 'graph_comparable_events':sum(graph_counts.values()),
 'missing_source_projects':unavailable_projects,
 'invalid_source_timestamps':bad_source_ts,
 'interpretation_limits':[
  'Source-to-graph event F1 is conditional on curated Jira data and fixed field mappings.',
  'Unmapped event categories are reported but not treated as incorrect supported-type predictions.',
  'Graph integrity is not event correctness.',
  'Extraction-to-graph retention is not independent semantic accuracy.',
  'Observed-at metadata does not certify leakage-free downstream feature queries.'
 ]}
(OUT/'full_graph_v2_report.json').write_text(json.dumps(report,indent=2,default=str),encoding='utf-8')
print('Saved outputs to:',OUT)
print('Source-to-graph exact F1:',None if independent is None else independent['F1'])
print('Extraction-to-graph retention F1:',None if not RESOLVED.is_file() else retention['F1'])
print('Reversed intervals:',interval_reversed,'suspicious observed-before-event:',obs_before)


Saved outputs to: /content/drive/MyDrive/01_Knowledge_Construction/Temporal_KG_Evaluation/Results/Full_Graph_Deterministic_V2
Source-to-graph exact F1: 0.4450143658767041
Extraction-to-graph retention F1: 0.4229307240419766
Reversed intervals: 35 suspicious observed-before-event: 68


## 10. Timestamp reconciliation — independent sensitivity audit

**Purpose.** Investigate why exact source-to-graph F1 differs from ±1-second F1. Preserve the original V2 outputs and scoring. Matching is one-to-one within `(instance_id, issue_key, event_type)`; never reuse a source or graph event. The tolerance metric is a **sensitivity analysis**, not proof of semantic equivalence. Original Jira changelog items and graph event payloads should still be checked for identity where available.

Outputs are saved to `Results/Full_Graph_Deterministic_V2/Timestamp_Reconciliation/`. Run after all preceding V2 cells. This section uses the existing `source_counts`, `graph_counts`, `independent`, `OUT`, and `metric` variables.


In [12]:
from collections import defaultdict, Counter
from pathlib import Path
import pandas as pd, numpy as np, json
AUDIT = OUT / 'Timestamp_Reconciliation'
AUDIT.mkdir(parents=True, exist_ok=True)
assert independent is not None, 'Run the earlier V2 cells first; source-to-graph F1 is unavailable.'
assert sum(source_counts.values()) > 0 and sum(graph_counts.values()) > 0

def grouped_event_times(counter):
    grouped=defaultdict(list)
    for (inst, issue, typ, timestamp), count in counter.items():
        t=pd.Timestamp(timestamp)
        if t.tzinfo is None: t=t.tz_localize('UTC')
        else: t=t.tz_convert('UTC')
        grouped[(inst,issue,typ)].extend([t]*int(count))
    return {key:sorted(times) for key,times in grouped.items()}

source_groups=grouped_event_times(source_counts)
graph_groups=grouped_event_times(graph_counts)
print('Source groups:',len(source_groups),'Graph groups:',len(graph_groups))
print('Source events:',sum(map(len,source_groups.values())), 'Graph events:',sum(map(len,graph_groups.values())))


Source groups: 9555 Graph groups: 5957
Source events: 32202 Graph events: 19657


### 10.1 Exact-first one-to-one reconciliation

Exact timestamp matches are reserved first. Remaining source/graph events are paired within one second, using chronological two-pointer matching. This prevents a graph event from being counted twice and allows us to inspect **only the newly recovered matches**. Exact-first is useful for auditing; the original V2 tolerance score is independently checked below.


In [13]:
def match_one_to_one(source_groups, graph_groups, tolerance_seconds=1.0):
    matches=[]; leftovers=[]
    for group in sorted(set(source_groups)|set(graph_groups)):
        source=list(source_groups.get(group,[]))
        graph=list(graph_groups.get(group,[]))
        # Reserve exact matches with multiset counts, preserving duplicates.
        sc=Counter(source); gc=Counter(graph)
        exact=sc & gc
        for t,count in sorted(exact.items()):
            for _ in range(count): matches.append((*group,t,t,0.0,'EXACT'))
        remaining_s=[];remaining_g=[]
        for t,count in sorted((sc-exact).items()): remaining_s.extend([t]*count)
        for t,count in sorted((gc-exact).items()): remaining_g.extend([t]*count)
        i=j=0; near_count=0
        while i<len(remaining_s) and j<len(remaining_g):
            delta=(remaining_g[j]-remaining_s[i]).total_seconds()
            if abs(delta)<=tolerance_seconds:
                matches.append((*group,remaining_s[i],remaining_g[j],delta,'WITHIN_1S_NOT_EXACT'))
                i+=1;j+=1;near_count+=1
            elif delta>tolerance_seconds: i+=1
            else: j+=1
        leftovers.append({'instance_id':group[0],'issue_key':group[1],'event_type':group[2],
                          'unmatched_source':len(remaining_s)-near_count,'unmatched_graph':len(remaining_g)-near_count})
    return matches,leftovers

matches,leftovers=match_one_to_one(source_groups,graph_groups)
matched=pd.DataFrame(matches,columns=['instance_id','issue_key','event_type','source_time','graph_time','graph_minus_source_seconds','match_class'])
matched['source_fractional_microseconds']=matched.source_time.map(lambda t:t.microsecond)
matched['graph_fractional_microseconds']=matched.graph_time.map(lambda t:t.microsecond)
matched['abs_delta_seconds']=matched.graph_minus_source_seconds.abs()
matched['graph_time_is_whole_second']=matched.graph_fractional_microseconds.eq(0)
matched['source_time_is_whole_second']=matched.source_fractional_microseconds.eq(0)
matched.to_csv(AUDIT/'one_to_one_timestamp_matches.csv',index=False)
pd.DataFrame(leftovers).to_csv(AUDIT/'unmatched_counts_by_issue_type.csv',index=False)
new=matched[matched.match_class=='WITHIN_1S_NOT_EXACT'].copy()
exact_n=int((matched.match_class=='EXACT').sum()); tolerant_n=len(matched)
print('Exact matches:',exact_n,'New matches within 1 second:',len(new),'Total one-to-one matches:',tolerant_n)
assert exact_n==independent['TP'], 'Exact matches disagree with V2: inspect duplicate or timestamp keys.'
assert tolerant_n<=min(sum(source_counts.values()),sum(graph_counts.values()))
print('Tolerance F1:',metric(tolerant_n,sum(graph_counts.values())-tolerant_n,sum(source_counts.values())-tolerant_n))


Exact matches: 11539 New matches within 1 second: 8117 Total one-to-one matches: 19656
Tolerance F1: {'TP': 19656, 'FP': 1, 'FN': 12546, 'precision': 0.9999491275372641, 'recall': 0.6103968697596422, 'F1': 0.7580554966351067}


### 10.2 Diagnose systematic timestamp precision changes

If most newly matched graph timestamps end at `.000000` while source timestamps contain fractional seconds, this supports a **timestamp precision-loss hypothesis**. It is not proof of a harmless conversion: compare original event IDs, changelog IDs, fields, and values before treating tolerant matches as factual matches.


In [14]:
if len(new):
    print('Absolute differences (seconds):')
    display(new.abs_delta_seconds.describe(percentiles=[.1,.25,.5,.75,.9,.95,.99]).to_frame('seconds'))
    print('Graph timestamps at whole seconds:',int(new.graph_time_is_whole_second.sum()),'/',len(new))
    print('Source timestamps with fractional seconds:',int((~new.source_time_is_whole_second).sum()),'/',len(new))
    print('Graph earlier than source:',int((new.graph_minus_source_seconds<0).sum()),'later:',int((new.graph_minus_source_seconds>0).sum()))
    display(new[['instance_id','issue_key','event_type','source_time','graph_time','graph_minus_source_seconds']].head(25))
    new['project']=new.issue_key.str.split('-').str[0]
    by_type=new.groupby('event_type').agg(new_matches=('event_type','size'),whole_second_graph=('graph_time_is_whole_second','sum'),median_abs_delta=('abs_delta_seconds','median')).reset_index()
    by_project=new.groupby('project').agg(new_matches=('project','size'),whole_second_graph=('graph_time_is_whole_second','sum'),median_abs_delta=('abs_delta_seconds','median')).reset_index()
    display(by_type);display(by_project)
    by_type.to_csv(AUDIT/'recovered_matches_by_type.csv',index=False)
    by_project.to_csv(AUDIT/'recovered_matches_by_project.csv',index=False)
else:
    print('No additional near-time matches found.')

# Important: show examples of event identity for manual source/graph spot-checks.
review=new.sort_values('abs_delta_seconds',ascending=False).head(40).copy()
review['review_decision']='UNREVIEWED'
review['review_note']=''
review.to_csv(AUDIT/'timestamp_identity_review_sample.csv',index=False)


Absolute differences (seconds):


,seconds
count,8117.000000
mean,0.501938
std,0.286642
min,0.001000
10%,0.105000
25%,0.255000
50%,0.503000
75%,0.749000
90%,0.896000
95%,0.949000


Graph timestamps at whole seconds: 8117 / 8117
Source timestamps with fractional seconds: 8117 / 8117
Graph earlier than source: 8117 later: 0


,instance_id,issue_key,event_type,source_time,graph_time,graph_minus_source_seconds
598,AMBARI-I01,AMBARI-1,ASSIGNEE_CHANGED,2016-08-10 23:29:36.480000+00:00,2016-08-10 23:29:36+00:00,-0.480
599,AMBARI-I01,AMBARI-1,ASSIGNEE_CHANGED,2016-08-10 23:30:02.026000+00:00,2016-08-10 23:30:02+00:00,-0.026
600,AMBARI-I01,AMBARI-10023,STATUS_CHANGED,2016-10-25 16:44:03.783000+00:00,2016-10-25 16:44:03+00:00,-0.783
601,AMBARI-I01,AMBARI-12263,ASSIGNEE_CHANGED,2016-07-15 21:53:22.251000+00:00,2016-07-15 21:53:22+00:00,-0.251
602,AMBARI-I01,AMBARI-12263,FIX_VERSION_CHANGED,2016-07-15 23:15:24.907000+00:00,2016-07-15 23:15:24+00:00,-0.907
603,AMBARI-I01,AMBARI-12263,STATUS_CHANGED,2016-07-15 23:15:24.907000+00:00,2016-07-15 23:15:24+00:00,-0.907
604,AMBARI-I01,AMBARI-12263,STATUS_CHANGED,2016-09-28 17:06:06.036000+00:00,2016-09-28 17:06:06+00:00,-0.036
605,AMBARI-I01,AMBARI-12263,STATUS_CHANGED,2016-09-28 17:07:00.921000+00:00,2016-09-28 17:07:00+00:00,-0.921
606,AMBARI-I01,AMBARI-12263,STATUS_CHANGED,2016-09-28 20:06:12.524000+00:00,2016-09-28 20:06:12+00:00,-0.524
607,AMBARI-I01,AMBARI-12263,STATUS_CHANGED,2016-09-28 20:07:04.581000+00:00,2016-09-28 20:07:04+00:00,-0.581


,event_type,new_matches,whole_second_graph,median_abs_delta
0,ASSIGNEE_CHANGED,808,808,0.5295
1,COMPONENT_CHANGED,509,509,0.4880
2,DUE_DATE_CHANGED,2,2,0.6110
3,FIX_VERSION_CHANGED,776,776,0.4840
4,PRIORITY_CHANGED,118,118,0.5450
5,STATUS_CHANGED,5904,5904,0.5020


,project,new_matches,whole_second_graph,median_abs_delta
0,AMBARI,817,817,0.4990
1,CONFSERVER,730,730,0.4720
2,DAOS,94,94,0.5730
3,FLEX,445,445,0.4760
4,HBASE,536,536,0.4835
5,INFRA,421,421,0.4740
6,JRACLOUD,544,544,0.5280
7,JRASERVER,267,267,0.5040
8,JSDCLOUD,466,466,0.4880
9,MC,1919,1919,0.5170


### 10.3 Tolerance sensitivity and audit summary

Exact F1 remains the primary baseline until the source-to-graph event identity and timestamp semantics are verified. The tolerant result is conditional on the fixed vocabulary and one-to-one timestamp matching. This audit does not test graph semantics, point-in-time feature leakage, or completeness of unsupported event categories.


In [15]:
total_source=sum(source_counts.values());total_graph=sum(graph_counts.values())
rows=[]
for label,tp in [('EXACT',exact_n),('PLUS_MINUS_1_SECOND_EXACT_FIRST',tolerant_n)]:
    rows.append({'matching_rule':label,'source_events':total_source,'graph_events':total_graph,**metric(tp,total_graph-tp,total_source-tp)})
score_table=pd.DataFrame(rows)
display(score_table)
score_table.to_csv(AUDIT/'timestamp_sensitivity_summary.csv',index=False)
summary={'status':'DIAGNOSTIC_ONLY','exact_tp':exact_n,'tolerant_tp':tolerant_n,
         'new_near_matches':len(new),'new_near_graph_whole_second':int(new.graph_time_is_whole_second.sum()) if len(new) else 0,
         'new_near_source_fractional':int((~new.source_time_is_whole_second).sum()) if len(new) else 0,
         'identity_verified':False,'limitations':['Matches are keyed by instance, issue, mapped event type, and timestamp only.',
         'Timestamp closeness does not independently establish identical changelog items.',
         'No reference/prediction records are edited; V2 scores are preserved.']}
(AUDIT/'timestamp_reconciliation_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('Audit saved:',AUDIT)
print('Next: inspect timestamp_identity_review_sample.csv against original Jira changelog IDs and graph event payloads.')


,matching_rule,source_events,graph_events,TP,FP,FN,precision,recall,F1
0,EXACT,32202,19657,11539,8118,20663,0.587017,0.358332,0.445014
1,PLUS_MINUS_1_SECOND_EXACT_FIRST,32202,19657,19656,1,12546,0.999949,0.610397,0.758055


Audit saved: /content/drive/MyDrive/01_Knowledge_Construction/Temporal_KG_Evaluation/Results/Full_Graph_Deterministic_V2/Timestamp_Reconciliation
Next: inspect timestamp_identity_review_sample.csv against original Jira changelog IDs and graph event payloads.
